# matmul-back-transpose-pair composite — cx26: matmul_back0 vs matmul_back1 — transpose pair, mirror argnums

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `matmul-back-transpose-pair`, `arg-position-back-functions`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "matmul-back-transpose-pair"
DD_ATOM_IDS = ["matmul-back-transpose-pair", "arg-position-back-functions"]
DD_SUBTOPICS = ["Backprop: matmul_back transpose pair", "Backprop: Arg-position back funcs"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA registers back-fns by `(func, argnum)`. For asymmetric ops like `@` (matmul), the back-fn is genuinely different per arg position. `matmul_back0` returns `dL/dx` for `x @ y`; `matmul_back1` returns `dL/dy`. They share grad_out but transpose the OTHER operand: `grad_out @ y.T` vs `x.T @ grad_out`. The arg-position-back-functions atom is what registers these distinct entries; the matmul-back-transpose-pair atom is the specific (x, y) ↔ (y.T, x.T) mirror that makes both shapes work out.

### Composite Exercise — matmul_back0 vs matmul_back1 — transpose pair, mirror argnums

**Atoms exercised together**: `matmul-back-transpose-pair`, `arg-position-back-functions`

Implement `cx26_matmul_back(grad_out, out, x, y, argnum)` that returns:

- if `argnum == 0`: `dL/dx` = `grad_out @ y.T`
- if `argnum == 1`: `dL/dy` = `x.T @ grad_out`
- otherwise: raise `ValueError`

One function, two formulas, dispatched by argnum — the same pattern ARENA uses inside its `back_funcs` registry.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx26_matmul_back(grad_out, out, x, y, argnum):
    raise NotImplementedError

def _test_cx26():
    x = t.randn(4, 3)
    y = t.randn(3, 5)
    out = x @ y
    grad_out = t.randn(4, 5)

    gx = cx26_matmul_back(grad_out, out, x, y, argnum=0)
    gy = cx26_matmul_back(grad_out, out, x, y, argnum=1)
    assert gx.shape == x.shape, f'gx shape: {gx.shape} vs {x.shape}'
    assert gy.shape == y.shape, f'gy shape: {gy.shape} vs {y.shape}'
    assert t.allclose(gx, grad_out @ y.T), 'gx formula wrong'
    assert t.allclose(gy, x.T @ grad_out), 'gy formula wrong'

    # Cross-check against torch autograd.
    xa = x.clone().requires_grad_(True)
    ya = y.clone().requires_grad_(True)
    (xa @ ya).backward(grad_out)
    assert t.allclose(gx, xa.grad), f'gx vs autograd mismatch'
    assert t.allclose(gy, ya.grad), f'gy vs autograd mismatch'

    # Bad argnum must raise.
    raised = False
    try: cx26_matmul_back(grad_out, out, x, y, argnum=2)
    except ValueError: raised = True
    assert raised, 'argnum=2 should raise ValueError'
    _dd_passed.add('cx26')

_test_cx26()

<details><summary>Show solution — cx26</summary>

```python
def cx26_matmul_back(grad_out, out, x, y, argnum):
    # Arg-position dispatch: same op, different formula per argnum.
    if argnum == 0:
        # dL/dx: (m,k) @ (k,n) shape works as grad_out @ y.T → (m,n).
        return grad_out @ y.T
    if argnum == 1:
        # dL/dy: x.T @ grad_out → (k,m) @ (m,n) = (k,n) = y.shape.
        return x.T @ grad_out
    raise ValueError(f'matmul has args (x, y); argnum must be 0 or 1, got {argnum}')
```

The transpose pair `(y.T, x.T)` is what makes both shapes line up. If you forgot the transpose, `grad_out @ y` would be a shape error, not a numerical one — the type system catches it. The argnum split exists because matmul is not commutative, so the back-fn genuinely needs two different code paths.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx26'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx26',
        'subtopics': ["Backprop: matmul_back transpose pair", "Backprop: Arg-position back funcs"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()